# **Synthetic Data Generation**
Due to data privacy restrictions in financial services, this project uses a synthetically generated dataset modeled on real-world digital lending behaviour in Kenya. 

In [3]:
import pandas as pd
import numpy as np
import random

In [4]:
n_rows = 50000
consumer_ratio = 0.75
sme_ratio = 0.25

## Consumer Details
### Create borrower type

In [5]:
borrower_type = np.random.choice(['Consumer', 'SME'], size= n_rows, p=[consumer_ratio, sme_ratio])

credit_score = np.random.randint(300, 851, size= n_rows)

### Generate borrower ages

In [6]:
# Create borrower ages
ages = [np.random.randint(18, 66) if bt == 'Consumer' else np.random.randint(25, 71) for bt in borrower_type]

### Assign counties


In [7]:
counties = [np.random.choice(['Nairobi', 'Mombasa', 'Kisumu', 'Nakuru', 'Machakos', 'Other'], p= [0.25, 0.1, 0.1, 0.1, 0.05, 0.4] if bt == 'Consumer' else [0.15, 0.1, 0.1, 0.15, 0.05, 0.45])
            for bt in borrower_type]

### Generate Income

In [8]:
income = [np.round(np.random.randint(10000, 150001)/1000)*1000 if bt=="Consumer" else np.round(np.random.randint(50000, 500001)/1000)*1000
          for bt in borrower_type]

In [9]:
df = pd.DataFrame({'borrower_type': borrower_type, 'credit_score': credit_score, 'age': ages, 'county': counties, 'income': income})

df.head(10)

,borrower_type,credit_score,age,county,income
0,Consumer,841,65,Nairobi,123000.0
1,Consumer,828,47,Nairobi,62000.0
2,Consumer,664,39,Other,72000.0
3,Consumer,461,22,Nairobi,70000.0
4,Consumer,453,46,Nairobi,113000.0
5,Consumer,569,34,Nakuru,17000.0
6,SME,805,43,Nairobi,184000.0
7,Consumer,364,40,Other,58000.0
8,Consumer,759,19,Nakuru,74000.0
9,SME,553,39,Other,482000.0


## Loan Details
### Loan amount

In [16]:
# Loan amount
df['loan_amount'] = np.where(
    df['borrower_type'] == 'Consumer',
    np.random.randint(5000, 200001, size=n_rows),
    np.random.randint(50000, 5000001, size=n_rows)
)

# Loan term in months
df['loan_term_months'] = np.where(
    df['borrower_type'] == 'Consumer',
    np.random.randint(6, 25, size=n_rows),
    np.random.randint(12, 61, size=n_rows)
)

# Loan Porpose
df['loan_purpose'] = np.where(
    df['borrower_type'] == 'Consumer',
    np.random.choice(["Education", "Medical", "Personal", "Business", "Emergency"], size= n_rows), 
    np.random.choice(["Working Capital", "Expansion", "Inventory", "Equipment"], size= n_rows)
    )

# Interest rate based on credit score
base_rate = np.where(df['borrower_type'] == 'Consumer', 12, 10)
df['interest_rate'] = base_rate + ((700 - df['credit_score']) / 50)
df['interest_rate'] = df['interest_rate'].clip(5, 25)

# Default probability
df['default_prob'] = np.where(
    df['borrower_type'] == 'Consumer',
    0.3 - ((df['credit_score'] - 300)/550)*0.25,
    0.25 - ((df['credit_score'] - 300)/550)*0.15
)
df['default_prob'] = df['default_prob'].clip(0, 1)

# Default outcome
df['defaulted'] = np.random.binomial(1, df['default_prob'])

In [17]:
df.head()

,borrower_type,credit_score,age,county,income,loan_amount,loan_term_months,interest_rate,default_prob,defaulted,loan_purpose
0,Consumer,841,65,Nairobi,123000.0,111984,13,9.18,0.054091,0,Medical
1,Consumer,828,47,Nairobi,62000.0,52512,24,9.44,0.060000,0,Business
2,Consumer,664,39,Other,72000.0,13480,18,12.72,0.134545,0,Emergency
3,Consumer,461,22,Nairobi,70000.0,132252,13,16.78,0.226818,0,Medical
4,Consumer,453,46,Nairobi,113000.0,75829,14,16.94,0.230455,0,Medical


In [19]:
df.to_csv('../Data/raw/loan_dataset.csv')